# v22 OOF Patch — append-only instrumentation

**What this does.** Turns a v22 fork into a leak-free out-of-fold scorer over the
773 training wells, and emits `v22_oof.pkl` (per-well error, predictions,
branch-spread, field-support distance) for the join against the BiGRU.

**What this does NOT do.** It does not modify your v22 model-code cell. Every
change here is a *redefinition in a later cell*, so the original definitions stay
byte-identical. Cell P1 is the one exception: it does verified string surgery on
`build_ufield`'s **source text**, asserts each anchor matches exactly once, and
prints a SHA-256 of the before/after source for your patch log.

**Where to put these cells.** Append them to the end of a **throwaway fork** of
v22. Run your v22 notebook top-to-bottom first (so `build_ufield`,
`_field_query`, `field_blend`, `predict_well_diag`, `load_well`, `wells`,
`CONFIG` all exist), then run P0 → P6 in order.

**Do not run v22's test-inference / submission cell in this fork.** This run is
for analysis only.

## Run order

| Cell | Purpose | Time |
|---|---|---|
| P0 | Preflight — assert required globals exist, snapshot hashes | instant |
| P1 | Patch `build_ufield` source: tag each field point with its well | instant |
| P2 | Redefine `_field_query`: exclude-then-build self-exclusion | instant |
| P3 | Wrap `field_blend` + `predict_well_diag`: branch spread + current-well | instant |
| P4 | Rebuild the field so the new `wid` tags take effect | ~40 s |
| P5 | **Validation gate** — prove self-exclusion is actually firing | ~10 s |
| P6 | OOF loop over 773 training wells → `v22_oof.pkl` | 45–75 min |

## The two gates that decide whether the output is trustworthy

1. **P5** must show the self-excluded nearest-neighbour distance jumping from a
   couple of feet to hundreds of feet, and the field-only blind MAE rising from
   ~0.2–0.4 (leaked) to roughly 13–15 (real). If it doesn't move, exclusion is
   not firing and every number downstream is fantasy.
2. **P6** must print a final OOF MAE near **~6.5**. That is v22's known local
   number. Materially above it means the field blend got switched off; materially
   below it means a leak survived.

In [ ]:
# ===== P0: preflight =====
# Asserts the v22 globals this patch depends on, and snapshots source hashes so
# the patch is auditable. Nothing is modified here.
import hashlib, inspect, sys
import numpy as np

_REQUIRED = ['build_ufield', '_field_query', 'field_blend',
             'predict_well_diag', 'load_well', 'wells', 'CONFIG']
_missing = [n for n in _REQUIRED if n not in globals()]
if _missing:
    raise NameError(
        'v22 globals not found: %s\n'
        'Run your v22 notebook top-to-bottom FIRST (model-code cell + the '
        'spatial-map/field cell), then run these patch cells.' % _missing)

def _srchash(fn):
    try:
        s = inspect.getsource(fn)
    except Exception:
        return ('<unavailable>', '')
    return (hashlib.sha256(s.encode()).hexdigest()[:16], s)

_PRE = {n: _srchash(globals()[n])[0] for n in _REQUIRED if callable(globals()[n])}
print('PREFLIGHT OK — all v22 globals present')
for k, v in _PRE.items():
    print('  %-20s src sha256[:16] = %s' % (k, v))

# re-entrancy guard state
_PATCH_APPLIED = globals().get('_PATCH_APPLIED', set())
print('\npatches already applied in this session:', sorted(_PATCH_APPLIED) or 'none')


## P1 — tag each field point with its well

`build_ufield` currently builds `wid = [''] * n_train`, so there is no way to tell
which field point came from which well, and self-exclusion is impossible. This
cell rewrites three anchors in the function's source:

- `pts = []; us = []` → also open a `widp` list
- the `pts.append(...)` / `us.append(...)` pair → tag every point with its well id
- `wid = [''] * n_train` → `wid = list(widp)`

plus one line stashing the raw point array into `_UFIELD_PTS`, which P2 needs to
rebuild trees. The `_UFIELD` dict literal is deliberately left alone.

Every replacement asserts `count == 1`. If an anchor doesn't match, the cell
prints the real source and stops rather than guessing.

In [ ]:
# ===== P1: patch build_ufield source (verified, exactly-once replacements) =====
import inspect, hashlib, textwrap

_src0 = inspect.getsource(build_ufield)
_h0 = hashlib.sha256(_src0.encode()).hexdigest()
print('build_ufield BEFORE sha256 =', _h0)

_src = textwrap.dedent(_src0)

# --- anchor 1: open the widp accumulator -------------------------------------
_A1_OLD = "pts = []; us = []"
_A1_NEW = "pts = []; us = []; widp = []"

# --- anchor 2: tag each appended point with its well id ----------------------
_A2_OLD = "pts.append(np.column_stack([h.X.values[m][::8], h.Y.values[m][::8]]))"
_A2_NEW = ("_p = np.column_stack([h.X.values[m][::8], h.Y.values[m][::8]])\n"
           "        pts.append(_p)\n"
           "        widp.extend([w] * len(_p))")

# --- anchor 3: use the real per-point well ids -------------------------------
_A3_OLD = "wid = [''] * n_train"
_A3_NEW = "wid = list(widp)"

# --- anchor 4: expose the raw point array for tree rebuilds ------------------
_A4_OLD = "P = np.vstack(pts); Uv = np.concatenate(us)"
_A4_NEW = ("P = np.vstack(pts); Uv = np.concatenate(us)\n"
           "    globals()['_UFIELD_PTS'] = P")

_FAILED = []
for _tag, _old in [('A1', _A1_OLD), ('A2', _A2_OLD), ('A3', _A3_OLD), ('A4', _A4_OLD)]:
    _n = _src.count(_old)
    print('  anchor %s: %d match(es)' % (_tag, _n))
    if _n != 1:
        _FAILED.append((_tag, _old, _n))

if _FAILED:
    print('\n' + '=' * 70)
    print('ANCHOR MISMATCH — patch NOT applied. Your v22 source differs from the')
    print('version this patch was written against. Full source below; send it to')
    print('Claude and the anchors will be re-cut against your actual text.')
    print('=' * 70)
    for _tag, _old, _n in _FAILED:
        print('  %s expected 1 match, found %d, for:\n    %r' % (_tag, _n, _old))
    print('-' * 70)
    print(_src0)
    raise SystemExit('P1 aborted: anchor mismatch (nothing was modified)')

for _old, _new in [(_A1_OLD, _A1_NEW), (_A2_OLD, _A2_NEW),
                   (_A3_OLD, _A3_NEW), (_A4_OLD, _A4_NEW)]:
    _src = _src.replace(_old, _new)

import ast
ast.parse(_src)                      # syntax gate before exec
exec(compile(_src, '<build_ufield_patched>', 'exec'), globals())

_h1 = hashlib.sha256(inspect.getsource(build_ufield).encode()).hexdigest()
print('\nbuild_ufield AFTER  sha256 =', _h1)
print('PATCH P1 APPLIED — field points now carry per-point well ids')
_PATCH_APPLIED.add('P1')
globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P2 — exclude-then-build self-exclusion

This is the subtle one, and it's the reason the obvious fix fails.

A well sits *on* its own dense lateral. With `field_sub=8`, a well contributes
roughly 820 field points spaced ~8 ft apart, spanning ±160 ft around any station
on it. So for any query point on that lateral, **all 40 nearest neighbours are the
well's own points** — the next well is hundreds of feet away.

Masking those neighbours *after* the k-nearest query therefore zeroes every
weight → `s ≈ 0` → `est = NaN` → the `okk.sum() < 50` guard trips → `field_blend`
returns `pred` untouched. You wouldn't get a leak. You'd get **v22 with the field
blend silently switched off**, scoring maybe 7.5–8 instead of ~6.55, and the join
would be comparing the BiGRU against a crippled v22.

The correct fix is to rebuild the KD-tree *without* the well's points, then query.
The current well is carried in the module-global `_CUR_WELL` (set by the
`predict_well_diag` wrapper in P3), so no call site anywhere in v22 has to change.

Note the `field_aug` branch is dropped from the query path — it's leaderboard-
falsified and switched off, so it was dead code. With `field_aug=False` and
`wid` previously all-`''`, the old function reduced to plain IDW with
`d_conf = dd[:,0]`, which is exactly what this computes. On test wells (which
aren't in the field, so `keep.all()` is True) this is bit-identical to v22.

In [ ]:
# ===== P2: exclude-then-build field query =====
from scipy.spatial import cKDTree

_CUR_WELL = globals().get('_CUR_WELL', None)   # set by the P3 wrapper
_EXCL_CACHE = {}

def _excluded_field(self_well):
    """Return (tree, U) with self_well's own points removed.

    Masking neighbours AFTER a shared-tree query is wrong: a well sits on its own
    dense lateral, so all k neighbours are its own points and masking zeroes every
    weight, silently disabling the blend. Rebuild without them instead.
    """
    F = _UFIELD
    if self_well is None:
        return F['tree'], F['U']
    hit = _EXCL_CACHE.get(self_well)
    if hit is not None:
        return hit
    keep = np.asarray(F['wid']) != self_well
    if keep.all():                      # test wells aren't in the field: no-op
        out = (F['tree'], F['U'])
    else:
        pts = globals().get('_UFIELD_PTS')
        if pts is None:
            raise RuntimeError('_UFIELD_PTS missing — re-run P1 then P4')
        out = (cKDTree(pts[keep]), np.asarray(F['U'])[keep])
    if len(_EXCL_CACHE) > 4:            # ~2 queries per well; 4 is plenty
        _EXCL_CACHE.pop(next(iter(_EXCL_CACHE)))
    _EXCL_CACHE[self_well] = out
    return out


def _field_query(xq, yq, self_well=None):
    F = _UFIELD
    k = int(CONFIG.get('field_k', 40))
    soft = float(CONFIG.get('field_soft', 400.0))
    if self_well is None:
        self_well = globals().get('_CUR_WELL', None)
    tree, U = _excluded_field(self_well)
    dd, idx = tree.query(np.column_stack([xq, yq]), k=k)
    wgt = 1.0 / (dd + soft) ** 2
    s = wgt.sum(1)
    est = np.einsum('nk,nk->n', wgt, U[idx]) / np.maximum(s, 1e-12)
    var = np.einsum('nk,nk->n', wgt,
                    (U[idx] - est[:, None]) ** 2) / np.maximum(s, 1e-12)
    est[s <= 1e-12] = np.nan
    return est, dd[:, 0], np.sqrt(np.maximum(var, 0))

print('PATCH P2 APPLIED — _field_query now excludes-then-builds, driven by _CUR_WELL')
_PATCH_APPLIED.add('P2')
globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P3 — branch spread + current-well plumbing

Two wrappers, both idempotent (guarded on `_orig_*` already existing, so
re-running this cell cannot double-wrap):

**`field_blend`** — v22 already builds `arr['_branch_paths']`, the per-branch
predictions whose disagreement drives the inverse-variance fusion. That spread
*is* the candidate routing signal for the join. The wrapper records the mean over
blind stations of the median-absolute-deviation across branch paths into
`diag['branch_spread_mean']`, then defers to the original. Signature is passed
through with `*a, **k`, so it adapts to however v22 calls it.

**`predict_well_diag`** — sets `_CUR_WELL` from `h.attrs['well']` for the duration
of the call, in a `try/finally` so it's always restored. This is what makes
self-exclusion fire without touching a single call site. The OOF loop in P6 sets
`h.attrs['well']` explicitly rather than relying on `load_well` doing it.

Anything that fails inside the spread capture is swallowed — a missing routing
diagnostic must never take down a prediction.

In [ ]:
# ===== P3: field_blend spread capture + predict_well_diag well plumbing =====

if '_orig_field_blend' not in globals():
    _orig_field_blend = field_blend

def field_blend(*a, **k):
    """Record per-well branch spread into diag, then defer to v22's original."""
    diag = k.get('diag');  arr = k.get('arr')
    if diag is None and len(a) >= 3:
        diag = a[2]
    if arr is None and len(a) >= 4:
        arr = a[3]
    try:
        if (arr is not None and isinstance(diag, dict)
                and arr.get('_branch_paths') is not None
                and len(arr['_branch_paths']) >= 2):
            _bp = np.stack(arr['_branch_paths'])
            _sp = np.median(np.abs(_bp - np.median(_bp, axis=0)), axis=0)
            _b = arr['blind']
            diag['branch_spread_mean'] = round(float(np.nanmean(_sp[_b])), 3)
    except Exception:
        pass                              # a diagnostic must never break a prediction
    return _orig_field_blend(*a, **k)


if '_orig_predict_well_diag' not in globals():
    _orig_predict_well_diag = predict_well_diag

def predict_well_diag(h, t, *a, **k):
    """Publish the current well to _CUR_WELL so _field_query can self-exclude."""
    global _CUR_WELL
    _prev = globals().get('_CUR_WELL', None)
    try:
        _CUR_WELL = h.attrs.get('well') if hasattr(h, 'attrs') else None
    except Exception:
        _CUR_WELL = None
    try:
        return _orig_predict_well_diag(h, t, *a, **k)
    finally:
        _CUR_WELL = _prev

print('PATCH P3 APPLIED — branch_spread_mean captured; _CUR_WELL plumbed')
print('  wrapped field_blend       :', _orig_field_blend)
print('  wrapped predict_well_diag :', _orig_predict_well_diag)
_PATCH_APPLIED.update(['P3'])
globals()['_PATCH_APPLIED'] = _PATCH_APPLIED


## P4 — rebuild the field

The field must be rebuilt so `_UFIELD['wid']` picks up the per-point well tags and
`_UFIELD_PTS` gets populated. Without this, P2 has nothing to exclude against and
will raise.

In [ ]:
# ===== P4: rebuild the structural field with tagged well ids =====
import time
_t0 = time.time()
_UFIELD = None
F = build_ufield()
print('field rebuilt in %.0fs' % (time.time() - _t0))

_wid = np.asarray(_UFIELD['wid'])
_n_unique = len(np.unique(_wid))
print('field points      : %d' % len(_UFIELD['U']))
print('unique well tags  : %d' % _n_unique)
print('_UFIELD_PTS shape : %s' % (np.shape(globals().get('_UFIELD_PTS')),))

assert _n_unique > 100, (
    'wid tagging did not take effect (%d unique tags). Re-run P1, then P4.' % _n_unique)
assert globals().get('_UFIELD_PTS') is not None, '_UFIELD_PTS missing — re-run P1 then P4'
_EXCL_CACHE.clear()
print('\nP4 OK — field carries per-point well ids')


## P5 — validation gate

This is the cell that decides whether anything below is worth reading. It runs the
field query on a single training well twice — once with self-exclusion off, once
with it on — and reports the nearest-neighbour distance and the field-only blind
MAE for each.

**Expected:**

| | nn_dist (median) | field-only blind MAE |
|---|---|---|
| self-**included** (leaked) | ~2 ft | ~0.2–0.4 |
| self-**excluded** (real) | hundreds of ft | ~13–15 |

The leaked MAE being near-zero is the whole point: the well is reading its own
answer off its own lateral. If the two rows look the same, exclusion is not
firing — stop and fix it before spending an hour on P6.

A third check matters just as much: the excluded estimate must not be mostly
`NaN`. All-NaN is the failure mode where the tree got rebuilt empty, and it's what
would silently switch the field blend off.

In [ ]:
# ===== P5: VALIDATION GATE — prove self-exclusion is firing =====
_tw = wells('train')
_w = _tw[100] if len(_tw) > 100 else _tw[0]
_h, _t = load_well('train', _w)
_h.attrs['well'] = _w

_z = _h['Z'].values.astype(float)
_tvt = _h['TVT'].values.astype(float)
_tin = _h['TVT_input'].values.astype(float)
_blind = ~np.isfinite(_tin)
_bok = _blind & np.isfinite(_tvt)
_ku = (~_blind) & np.isfinite(_tin)

print('gate well: %s   blind stations: %d   known: %d\n' % (_w, _bok.sum(), _ku.sum()))
print('%-16s %12s %12s %10s' % ('mode', 'nn_dist(med)', 'blind MAE', 'finite%'))
print('-' * 54)

_res = {}
for _tag, _sw in [('self-included', None), ('self-excluded', _w)]:
    _EXCL_CACHE.clear()
    _est, _dmin, _ = _field_query(_h.X.values, _h.Y.values, self_well=_sw)
    _fin = np.isfinite(_est)
    _ok = _ku & _fin
    if _ok.sum() < 10:
        print('%-16s %12s %12s %9.1f%%' % (_tag, 'n/a', 'NO ANCHOR', 100 * _fin.mean()))
        _res[_tag] = (np.nan, np.nan)
        continue
    _off = np.median((_tin + _z)[_ok] - _est[_ok])
    _ftvt = _est + _off - _z
    _m = _bok & np.isfinite(_ftvt)
    _mae = float(np.abs(_ftvt[_m] - _tvt[_m]).mean())
    _nn = float(np.median(_dmin[_bok]))
    _res[_tag] = (_nn, _mae)
    print('%-16s %12.0f %12.2f %9.1f%%' % (_tag, _nn, _mae, 100 * _fin.mean()))

_EXCL_CACHE.clear()
_nn_in, _mae_in = _res['self-included']
_nn_ex, _mae_ex = _res['self-excluded']

print('\n' + '=' * 54)
_pass = True
if not np.isfinite(_mae_ex):
    print('FAIL: excluded query produced no usable anchor (all-NaN field).')
    print('      This is the "field blend silently off" failure mode.')
    _pass = False
if np.isfinite(_nn_ex) and np.isfinite(_nn_in) and _nn_ex < 10 * max(_nn_in, 1.0):
    print('FAIL: nn_dist barely moved (%.0f -> %.0f). Exclusion is not firing.'
          % (_nn_in, _nn_ex))
    _pass = False
if np.isfinite(_mae_ex) and _mae_ex < 3.0:
    print('FAIL: excluded blind MAE %.2f is implausibly low — a leak survived.' % _mae_ex)
    _pass = False
if _pass:
    print('GATE PASSED — self-exclusion is real.')
    print('  leaked MAE %.2f (nn %.0f ft)  ->  honest MAE %.2f (nn %.0f ft)'
          % (_mae_in, _nn_in, _mae_ex, _nn_ex))
    print('  Honest MAE should sit near 13-15. Proceed to P6.')
else:
    print('\nDO NOT RUN P6 until this passes.')
print('=' * 54)


## P6 — the OOF loop

Scores v22 over all 773 training wells with each well excluded from its own field,
which mirrors deployment exactly: a test well is never in the field.

Output `v22_oof.pkl` carries four things, keyed by well:

- `err` — per-well blind-zone MAE in **absolute TVT space**
- `pred` — the blind station indices and predicted TVT (for the join, and for any
  blend you want to evaluate offline)
- `spread` — `branch_spread_mean`, routing candidate #1
- `nn` — `nn_dist`, field-support distance, routing candidate #2

**The comparability trap.** The BiGRU trains on the increment target with its own
per-well datum and `u_last`; v22 has its own anchor logic. The two are only
comparable if both errors are `abs(pred_TVT − true_TVT)` over the *identical*
blind mask of the *identical* wells. This loop is written in absolute TVT space
for exactly that reason — make sure the BiGRU side is too, or you'll chase a
phantom.

**The validity gate.** Final printed OOF MAE should land near **~6.5**. Around
7.5–8 means the field blend is off (P5 would normally have caught it). Well below
6 means a leak survived.

Budget 45–75 minutes: v22 is roughly 3.6 s/well, plus a tree rebuild per well.
Checkpoints are written every 50 wells, so a timeout doesn't cost you the run.

In [ ]:
# ===== P6: v22 OOF over the training wells (leak-free via self-exclusion) =====
import pickle, time, traceback

assert 'P1' in _PATCH_APPLIED and 'P2' in _PATCH_APPLIED and 'P3' in _PATCH_APPLIED, \
    'run P1-P3 first (applied: %s)' % sorted(_PATCH_APPLIED)

tr_wells = wells('train')
v22_err, v22_pred, v22_spread, v22_nn = {}, {}, {}, {}
skipped = []
t0 = time.time()

for i, w in enumerate(tr_wells):
    try:
        h, t = load_well('train', w)
        h.attrs['well'] = w                    # -> _CUR_WELL -> self-exclusion active

        truth = h['TVT'].values.astype(float)
        blind = ~np.isfinite(h['TVT_input'].values.astype(float))
        m = blind & np.isfinite(truth)
        if m.sum() < 1:
            skipped.append((w, 'no scorable blind stations'))
            continue

        pred, status, diag = predict_well_diag(h, t)
        pred = np.asarray(pred, dtype=float)

        good = m & np.isfinite(pred)
        if good.sum() < 1:
            skipped.append((w, 'all predictions NaN'))
            continue

        v22_err[w] = float(np.abs(pred[good] - truth[good]).mean())
        v22_pred[w] = dict(blind_idx=np.where(good)[0].astype(np.int32),
                           tvt_hat=pred[good].astype(np.float32))
        v22_spread[w] = float(diag.get('branch_spread_mean', np.nan))
        v22_nn[w] = float(diag.get('nn_dist', np.nan))

    except Exception as e:
        skipped.append((w, repr(e)[:120]))
        if len(skipped) <= 3:
            traceback.print_exc()

    if (i + 1) % 50 == 0:
        _run = np.mean(list(v22_err.values())) if v22_err else np.nan
        _el = time.time() - t0
        _eta = _el / (i + 1) * (len(tr_wells) - i - 1)
        print('%4d/%d  running mean err %.3f  skipped %d  [%.0fs elapsed, ~%.0fs left]'
              % (i + 1, len(tr_wells), _run, len(skipped), _el, _eta), flush=True)
        pickle.dump({'err': v22_err, 'pred': v22_pred, 'spread': v22_spread,
                     'nn': v22_nn, 'partial': True},
                    open('v22_oof.pkl', 'wb'))

pickle.dump({'err': v22_err, 'pred': v22_pred, 'spread': v22_spread,
             'nn': v22_nn, 'partial': False},
            open('v22_oof.pkl', 'wb'))

_oof = float(np.mean(list(v22_err.values())))
_errs = np.array(list(v22_err.values()))
print('\n' + '=' * 62)
print('saved v22_oof.pkl | v22 OOF MAE %.3f over %d wells (%.0f min)'
      % (_oof, len(v22_err), (time.time() - t0) / 60))
print('  per-well err: p10 %.2f  median %.2f  p90 %.2f  max %.2f'
      % tuple(np.percentile(_errs, [10, 50, 90]).tolist() + [_errs.max()]))
_sp = np.array([v22_spread[w] for w in v22_err]); _nn = np.array([v22_nn[w] for w in v22_err])
print('  branch_spread_mean finite: %d/%d' % (np.isfinite(_sp).sum(), len(_sp)))
print('  nn_dist finite           : %d/%d' % (np.isfinite(_nn).sum(), len(_nn)))
if skipped:
    print('  skipped %d wells; first few: %s' % (len(skipped), skipped[:5]))

print('-' * 62)
if 6.0 <= _oof <= 7.2:
    print('VALIDITY GATE PASSED — %.3f is in range of v22\'s known local ~6.5.' % _oof)
elif _oof > 7.2:
    print('GATE FAILED (%.3f too high) — the field blend is likely switched off.' % _oof)
    print('  Check P5 passed, and that build_ufield/_field_query were re-run in order.')
else:
    print('GATE FAILED (%.3f too low) — a leak likely survived. Re-check P5.' % _oof)
print('=' * 62)
print('\nDownload v22_oof.pkl from the Output tab, then run the join against')
print('bigru_oof.pkl. Treat anything under ~0.2 local improvement as noise.')


## Notes

**Do not run v22's submission cell in this fork.** `field_blend` and
`predict_well_diag` are wrapped here, and `build_ufield` has been rewritten in
memory. The wrappers are behaviour-preserving on test wells (`_CUR_WELL` is
`None`, `keep.all()` is True, so the query is bit-identical), but there is no
reason to take the risk on a scoring run. Keep the real v22 sealed.

**Re-running cells is safe.** P1 asserts exactly-once anchors before touching
anything. P2 is a plain redefinition. P3 guards on `_orig_*` so it cannot
double-wrap. P4 clears `_EXCL_CACHE`.

**If P1 aborts on an anchor mismatch**, it prints your actual `build_ufield`
source. Send that source over and the anchors get re-cut against your real text —
that's a two-minute fix, and it's exactly why the cell refuses to guess.

**Reading the join.** Across v19/v22/v23/v24 the local and leaderboard orderings
came out perfectly inverted, with spreads of 0.04 local and 0.02 LB. That's almost
certainly noise rather than genuine anti-correlation, but the implication holds
either way: local differences at the 0.04 scale carry no predictive signal for the
leaderboard. Don't ship on a 0.05.

**One asymmetry worth keeping in mind, and it favours you.** The BiGRU's OOF is a
true 5-fold held-out number. v22's OOF here benefits from a field built over all
training wells minus self, which is slightly more favourable to v22. So if a blend
wins in this comparison, it should win by at least as much on the leaderboard,
not less.